# Lección 2 — Clasificación vs Regresión

**Tema:** la misma data, dos preguntas distintas — cómo el TIPO de pregunta define el tipo de problema de Machine Learning.

**Objetivos:**
- Distinguir intuitivamente cuándo un problema es de **clasificación** (predice una categoría) o de **regresión** (predice un número).
- Ver código real de ML corriendo, sin necesidad de entender cada línea — el foco está en el OUTPUT.

**Cómo usar este notebook:** ejecutar cada celda con Shift+Enter para ver los outputs. El foco está en entender qué hace el modelo, no en escribir código.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EduPav/ai-learning-notebooks/blob/main/lessons/02-clasificacion-regresion/notebook.ipynb)

## Cómo usar este notebook

- **Convención de celdas:** todas las celdas de código son **Demo** — podés ejecutarlas y simplemente leer el resultado. Los **Mini-ejercicios** son de observación: mirás el output y respondés la pregunta mentalmente.
- **Aviso:** este notebook usa `pandas`, `numpy` y `sklearn`. Las vas a aprender en lecciones próximas. Por ahora, lo importante es VER qué pasa cuando corren.

In [ ]:
# Setup — ejecutar una sola vez al inicio.
# Todas las librerías que usamos vienen pre-instaladas en Google Colab,
# así que NO hace falta hacer pip install.

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, mean_squared_error

# Fijamos una seed para que los resultados sean reproducibles.
np.random.seed(42)

print("Setup completo. Todo listo para correr la demo.")

## Mismo dataset, dos preguntas: clasificación vs regresión con Iris

**Idea central:** vamos a usar **el mismo dataset** (Iris, un dataset clásico con medidas de 150 flores de 3 especies) para responder DOS preguntas distintas:

![iris_dataset.png](./iris_dataset.png)

1. **CASO A — Clasificación:** dada una flor, ¿de qué **especie** es? (predice una categoría: setosa / versicolor / virginica)
2. **CASO B — Regresión:** dada una flor, ¿cuánto mide su **pétalo**? (predice un número: cm)

El dataset es exactamente el mismo. Lo único que cambia es la PREGUNTA. Y eso decide si es clasificación o regresión.

Veamos primero cómo se ven los datos.

In [ ]:
# Demo — el profesor ejecuta y explica.
# Cargamos el dataset Iris y miramos las primeras 8 filas.

iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["especie"] = [iris.target_names[i] for i in iris.target]

print("Dimensiones del dataset:", df.shape, "→ 150 flores, 4 medidas + especie")
df.head(8)

Cada fila es una flor. Las 4 primeras columnas son medidas en centímetros (largo y ancho de sépalo y pétalo). La última columna es la **especie**, que es lo que el experto humano identificó mirando la flor.

Ahora sí: dos preguntas distintas, dos modelos distintos, mismo dataset.

### CASO A — Clasificación: ¿de qué especie es esta flor?

Acá el modelo recibe las 4 medidas y devuelve una **categoría**: setosa, versicolor o virginica.

In [ ]:
# Demo
# Entrenamos un clasificador con las 4 medidas para predecir la especie.

X = iris.data
y_clasificacion = iris.target  # 0, 1, 2 → setosa, versicolor, virginica

X_train, X_test, y_train, y_test = train_test_split(
    X, y_clasificacion, test_size=0.3, random_state=42
)

modelo_clasificacion = LogisticRegression(max_iter=1000, random_state=42)
modelo_clasificacion.fit(X_train, y_train)

y_pred = modelo_clasificacion.predict(X_test)

# Mostramos predicciones vs reales en una tabla comparativa.
tabla = pd.DataFrame({
    "especie_real": [iris.target_names[i] for i in y_test[:12]],
    "especie_predicha": [iris.target_names[i] for i in y_pred[:12]],
    "acerto": ["si" if a == b else "NO" for a, b in zip(y_test[:12], y_pred[:12])],
})
tabla

**¿Qué mirar?**

- La columna `especie_real` es la respuesta verdadera (la que dijo el experto humano).
- La columna `especie_predicha` es lo que dijo el modelo.
- La columna `acerto` te dice si el modelo le pegó o no.

El output del modelo es una **etiqueta** (texto de una categoría fija). Eso es **clasificación**.

### Mini-ejercicio — Clasificación

Mirá la tabla de arriba.

1. ¿Hay alguna fila donde el modelo haya fallado (columna `acerto` = "NO")?
2. Si la hay: ¿qué especie predijo el modelo? ¿Cuál era la especie real?

--SPOILER ABAJO--

**Respuesta esperada:** el modelo acertó en todas.

### CASO B — Regresión: ¿cuánto mide el pétalo de esta flor?

Mismo dataset. Pero ahora el modelo recibe 3 medidas (sépalo largo, sépalo ancho, pétalo ancho) y debe predecir un **número**: el largo del pétalo en centímetros.

In [ ]:
# Demo
# Mismo dataset, otra pregunta. Ahora predecimos un número.

X_regr = iris.data[:, [0, 1, 3]]  # sepal_length, sepal_width, petal_width
y_regresion = iris.data[:, 2]  # petal_length, lo que queremos predecir (cm)

X_train, X_test, y_train, y_test = train_test_split(
    X_regr, y_regresion, test_size=0.3, random_state=42
)

modelo_regresion = LinearRegression()
modelo_regresion.fit(X_train, y_train)

y_pred = modelo_regresion.predict(X_test)

tabla = pd.DataFrame({
    "largo_petalo_real_cm": np.round(y_test[:12], 2),
    "largo_petalo_predicho_cm": np.round(y_pred[:12], 2),
    "error_cm": np.round(y_pred[:12] - y_test[:12], 2),
})
tabla

**¿Qué mirar?**

- La columna `largo_petalo_real_cm` es el largo verdadero del pétalo (lo que se midió).
- La columna `largo_petalo_predicho_cm` es lo que el modelo estimó.
- La columna `error_cm` te dice por cuántos centímetros se equivocó el modelo en cada flor.

Ahora no hay categorías. El output es un número continuo (puede ser 1.4, 4.7, 5.83, lo que sea). Eso es **regresión**.

Ya no preguntamos "¿acertó o no?" sino "¿qué tan lejos estuvo?".

### Mini-ejercicio — Regresión

Mirá la columna `error_cm` de la tabla de arriba.

1. ¿Cuál es el valor absoluto (magnitud, sin importar si el número es positivo o negativo) de la predicción con mayor error?
2. ¿Por cuántos centímetros se equivocó el modelo en esa flor?
3. ¿Cuál es el valor del largo del pétalo que debería haber predicho el modelo si fuera perfecto?

--SPOILER ABAJO--

**Respuesta esperada:** 
1. 1,04
2. 1,04cm
3. 5,1cm

## Cierre del notebook

**Lo que vimos:**

- El TIPO de problema (clasificación o regresión) NO depende del dataset, depende de la PREGUNTA que le hacés.
- En clasificación, el modelo devuelve una **categoría** y se puede evaluar con "¿acertó o no?".
- En regresión, el modelo devuelve un **número** y se puede evaluar con "¿qué tan lejos estuvo?".

**En la próxima lección:** vamos a aprender las distintas formas de MEDIR si un modelo tiene buena performance.